# HW — Topic 9 · 상관계수에 속지 않기

**확률통계 · Topic 9** | 배점 25점 | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`HW_학번_T09.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T09.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| 1 | 나만의 "같은 통계량, 다른 그림" 만들기 | 10 |
| 2 | 상관계수 vs 실제 종속성 | 8 |
| 3 | 상관 ≠ 인과 사례 분석 | 7 |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −2점.
채점 루브릭과 자주 하는 실수는 [T09_hw.md](T09_hw.md) 를 참고한다.

## Part 0. 준비

이 셀을 먼저 실행한다. **시드는 바꾸지 말 것.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)   # ⚠️ 이 줄은 바꾸지 마세요


def normalize(x, y):
    """평균 0, 표준편차 1, 상관 0 으로 맞춘다."""
    x = (x - x.mean()) / x.std()
    y = (y - y.mean()) / y.std()
    r = np.corrcoef(x, y)[0, 1]
    y = y - r * x
    y = (y - y.mean()) / y.std()
    return x, y


print("준비 완료")

## 문제 1 — 나만의 "같은 통계량, 다른 그림" 만들기 (10점)

평균·표준편차·상관계수가 **모두 같은데 그림은 전혀 다른** 데이터셋 **3개**를 만든다.
`normalize()` 를 쓰면 평균 0·표준편차 1·상관 0 으로 강제로 맞출 수 있다.

> 💡 아이디어: 두 개의 직선(X자), 나선, 격자, 하트 모양, 글자 모양…
> 수식으로 만들기 어렵다면 **점을 손으로 몇십 개 찍어 배열로 적어도 된다.**

### ✏️ TODO 1 — 세 데이터 생성 (a) · 3점

아래는 예시로 **"두 개의 직선(X자)"** 을 만든다. 여러분은 **나머지 두 모양**을
직접 설계해서 `datasets` 딕셔너리에 추가하세요 (이름은 자유).

In [ ]:
N = 1000

# 모양 1: 두 개의 직선이 X자로 교차 (예시 — 그대로 둔다)
t = rng.uniform(-1, 1, N)
sign = rng.choice([-1.0, 1.0], N)
x1, y1 = normalize(t, sign * t)

# TODO 1: 나머지 두 모양을 직접 설계하세요 (나선, 격자, 하트, 글자 등 자유)
#         힌트 - 원하는 모양의 (x, y) 원본 배열을 만든 뒤 normalize(x, y) 를 씌우면 된다
x2, y2 = normalize(rng.normal(size=N), rng.normal(size=N))   # <- 이 두 줄을 고치세요
x3, y3 = normalize(rng.normal(size=N), rng.normal(size=N))   # <- 이 두 줄을 고치세요

datasets = {
    "Two lines": (x1, y1),
    "Shape 2": (x2, y2),
    "Shape 3": (x3, y3),
}
print(f"{len(datasets)}개 데이터셋 준비 완료:", list(datasets.keys()))

### ✏️ TODO 2 — 통계량을 표로 출력 (b) · 3점

세 데이터의 평균·표준편차·상관계수를 표로 출력해 **모두 같음**을 보인다.

In [ ]:
# TODO 2: 각 데이터셋의 평균·표준편차·상관계수를 출력하세요
#         힌트 - x.mean(), x.std(), np.corrcoef(x, y)[0, 1]
print(f"{'name':<12}{'mean x':>9}{'mean y':>9}{'std x':>9}{'std y':>9}{'corr':>9}")
for name, (x, y) in datasets.items():
    print(f"{name:<12}{0.0:9.3f}{0.0:9.3f}{0.0:9.3f}{0.0:9.3f}{0.0:9.3f}")   # <- 이 줄을 고치세요

### ✏️ TODO 3 — 세 산점도를 나란히 (c) · 2점

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(4 * len(datasets), 4))
for ax, (name, (x, y)) in zip(axes, datasets.items()):
    # TODO 3: 산점도를 그리세요
    #         힌트 - ax.plot(x, y, ".")
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_aspect("equal")
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

### (d) 이 실험이 주는 교훈 — 2문장

위 표와 그림을 **근거로** 쓴다. 이 셀을 더블클릭해서 작성한다.

> **답:** (여기에 작성)

## 문제 2 — 상관계수 vs 실제 종속성 (8점)

아래 세 관계에 대해 각각 표본 2,000개를 만들고 분석한다.

| 관계 | 정의 |
|---|---|
| A | $Y = 2X + \varepsilon$, $\varepsilon \sim N(0, 1)$ |
| B | $Y = X^2 + \varepsilon$ |
| C | $Y = \sin(3X) + \varepsilon$ |

($X \sim U(-2, 2)$, $\varepsilon \sim N(0, 0.3^2)$)

### ✏️ TODO 4 — 세 관계의 표본과 상관계수 (a) · 3점

In [ ]:
M = 2000
x_rel = rng.uniform(-2, 2, M)
eps = rng.normal(0, 0.3, M)

# TODO 4: A, B, C 세 관계의 y를 정의하고 상관계수를 구하세요
#         힌트 - A: 2*x_rel + rng.normal(0,1,M) / B: x_rel**2 + eps / C: np.sin(3*x_rel) + eps
relations = {
    "A: y=2x+noise": (x_rel, np.zeros(M)),
    "B: y=x^2+noise": (x_rel, np.zeros(M)),
    "C: y=sin(3x)+noise": (x_rel, np.zeros(M)),
}

print(f"{'relation':<20}{'corr':>9}")
for name, (x, y) in relations.items():
    r = np.corrcoef(x, y)[0, 1]
    print(f"{name:<20}{r:9.3f}")

### ✏️ TODO 5 — 세 산점도 (b) · 2점

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (name, (x, y)) in zip(axes, relations.items()):
    # TODO 5: 산점도를 그리세요
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

### (c) 상관계수와 예측 가능성의 괴리 — 3점

**"$X$ 를 알면 $Y$ 를 얼마나 잘 예측할 수 있는가"** 를 상관계수와 비교해 설명한다.
상관계수가 낮은데도 예측이 잘 되는 경우가 있는가? 위 표·그림을 근거로 쓴다.

> **답:** (여기에 작성)

## 문제 3 — 상관 ≠ 인과 사례 분석 (7점)

**실제로 보도된 "상관관계" 기사나 주장**을 하나 찾아 분석한다.
(뉴스, 논문 초록, SNS 게시물, 광고 문구 등 무엇이든)

⚠️ 특정 인물·집단을 비난하는 내용은 피할 것. **주장의 논리 구조**만 분석한다.

### (a) 어떤 두 변수의 상관을 주장하는가 — 2점

출처 링크를 함께 적는다.

> **답:** (여기에 작성 — 출처 링크 포함)

### (b) 그 주장이 인과를 암시하는가 — 1점

어떤 표현에서 그렇게 읽히는지 쓴다.

> **답:** (여기에 작성)

### (c) 제3의 변수(교란변수)로 설명 가능한 시나리오 — 2점

> **답:** (여기에 작성)

### (d) 인과를 확인하려면 어떤 실험/데이터가 필요한가 — 2점

> **답:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] `TODO 1~5` 를 모두 채웠다
- [ ] 문제 1(d) · 2(c) · 3(a)~(d) 의 **서술을 작성했다**
- [ ] 3(a)에 **출처 링크**를 적었다
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 그래프에 축 이름 · 제목이 있고 **한글이 없다**
- [ ] 파일명을 **`HW_학번_T09.ipynb`** 로 바꿨다

**제출처와 기한, 채점 루브릭은 [T09_hw.md](T09_hw.md) · PLATO · Google Classroom 공지를 확인한다.**